# Part 4 — Algorithm Visualization
**Algorithm Analysis and Simulation Toolkit | Term 2, SY 2025–2026**

Run cells top-to-bottom. Each cell produces one chart saved to `/tmp/`.

| Cell | Chart |
|------|-------|
| **Cell 1** | Setup — algorithms defined here |
| **Cell 2** | Chart 1: Sorting Benchmark Bars (time) |
| **Cell 3** | Chart 2: Sorting Scalability Curve |
| **Cell 4** | Chart 3: Best / Average / Worst Heatmap |
| **Cell 5** | Chart 4: MST Graph Visualization |
| **Cell 6** | Chart 5: Recursive Call Count Growth |
| **Cell 7** | Chart 6: Tower of Hanoi Exponential Growth |
| **Cell 8** | Chart 7: Recursion Tree Diagrams *(bonus)* |
| **Cell 9** | Chart 8: Sorting Metrics Breakdown (comps + swaps) |
| **Cell 10** | Chart 9: MST Cost Comparison (10 random graphs) |

In [ ]:
# ═══════════════════════════════════════════════════════════════════════
#  CELL 1 — SETUP
#  All algorithms defined here; reused by every chart below.
# ═══════════════════════════════════════════════════════════════════════
import time, random, heapq, math
from collections import defaultdict
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import networkx as nx
import numpy as np

plt.rcParams.update({
    "figure.dpi": 110,
    "font.family": "monospace",
    "axes.titlesize": 12,
    "axes.titleweight": "bold",
    "axes.labelsize": 10,
})
COLORS = ["#E63946","#457B9D","#2A9D8F","#E9C46A","#F4A261","#264653","#A8DADC","#9B5DE5"]

random.seed(42)
np.random.seed(42)

# ── Sorting ──────────────────────────────────────────────────────────
def bubble_sort(arr):
    a=arr[:]; n=len(a); c=s=0
    for i in range(n):
        sw=False
        for j in range(n-i-1):
            c+=1
            if a[j]>a[j+1]: a[j],a[j+1]=a[j+1],a[j]; s+=1; sw=True
        if not sw: break
    return a,c,s

def selection_sort(arr):
    a=arr[:]; n=len(a); c=s=0
    for i in range(n):
        mi=i
        for j in range(i+1,n):
            c+=1
            if a[j]<a[mi]: mi=j
        if mi!=i: a[i],a[mi]=a[mi],a[i]; s+=1
    return a,c,s

def insertion_sort(arr):
    a=arr[:]; c=s=0
    for i in range(1,len(a)):
        key=a[i]; j=i-1
        while j>=0:
            c+=1
            if a[j]>key: a[j+1]=a[j]; s+=1; j-=1
            else: break
        a[j+1]=key
    return a,c,s

def merge_sort(arr):
    comps=[0]
    def mg(L,R):
        o=[]; i=j=0
        while i<len(L) and j<len(R):
            comps[0]+=1
            if L[i]<=R[j]: o.append(L[i]); i+=1
            else: o.append(R[j]); j+=1
        o.extend(L[i:]); o.extend(R[j:]); return o
    def s(a):
        if len(a)<=1: return a
        m=len(a)//2; return mg(s(a[:m]),s(a[m:]))
    return s(arr[:]),comps[0],0

def quick_sort(arr):
    c=[0]; s=[0]
    def p(a,lo,hi):
        pv=a[hi]; i=lo-1
        for j in range(lo,hi):
            c[0]+=1
            if a[j]<=pv: i+=1; a[i],a[j]=a[j],a[i]; s[0]+=1
        a[i+1],a[hi]=a[hi],a[i+1]; s[0]+=1; return i+1
    def q(a,lo,hi):
        if lo<hi: pp=p(a,lo,hi); q(a,lo,pp-1); q(a,pp+1,hi)
    a=arr[:]; q(a,0,len(a)-1); return a,c[0],s[0]

def random_quick_sort(arr):
    c=[0]; s=[0]
    def p(a,lo,hi):
        r=random.randint(lo,hi); a[r],a[hi]=a[hi],a[r]; s[0]+=1
        pv=a[hi]; i=lo-1
        for j in range(lo,hi):
            c[0]+=1
            if a[j]<=pv: i+=1; a[i],a[j]=a[j],a[i]; s[0]+=1
        a[i+1],a[hi]=a[hi],a[i+1]; s[0]+=1; return i+1
    def q(a,lo,hi):
        if lo<hi: pp=p(a,lo,hi); q(a,lo,pp-1); q(a,pp+1,hi)
    a=arr[:]; q(a,0,len(a)-1); return a,c[0],s[0]

def counting_sort(arr):
    if not arr: return [],0,0
    a=arr[:]; mx=max(a); cnt=[0]*(mx+1); c=0
    for v in a: cnt[v]+=1; c+=1
    return [i for i,cc in enumerate(cnt) for _ in range(cc)],c,0

def radix_sort(arr):
    if not arr: return [],0,0
    a=arr[:]; c=0; exp=1; mx=max(a)
    while mx//exp>0:
        b=[[] for _ in range(10)]
        for n in a: b[(n//exp)%10].append(n); c+=1
        a=[x for bk in b for x in bk]; exp*=10
    return a,c,0

ALGORITHMS = [
    ("Bubble Sort",       bubble_sort),
    ("Selection Sort",    selection_sort),
    ("Insertion Sort",    insertion_sort),
    ("Merge Sort",        merge_sort),
    ("Quick Sort",        quick_sort),
    ("Random-Quick Sort", random_quick_sort),
    ("Counting Sort",     counting_sort),
    ("Radix Sort",        radix_sort),
]

# ── MST ──────────────────────────────────────────────────────────────
class UF:
    def __init__(self,n): self.p=list(range(n)); self.r=[0]*n
    def find(self,x):
        if self.p[x]!=x: self.p[x]=self.find(self.p[x])
        return self.p[x]
    def union(self,x,y):
        rx,ry=self.find(x),self.find(y)
        if rx==ry: return False
        if self.r[rx]<self.r[ry]: rx,ry=ry,rx
        self.p[ry]=rx
        if self.r[rx]==self.r[ry]: self.r[rx]+=1
        return True

def kruskal(verts, edges):
    vi={v:i for i,v in enumerate(verts)}; uf=UF(len(verts)); mst=[]
    for u,v,w in sorted(edges,key=lambda e:e[2]):
        if uf.union(vi[u],vi[v]): mst.append((u,v,w))
        if len(mst)==len(verts)-1: break
    return mst, sum(w for _,_,w in mst)

def prim(verts, edges, start=None):
    adj=defaultdict(list)
    for u,v,w in edges: adj[u].append((v,w)); adj[v].append((u,w))
    s=verts[0] if start is None else start
    vis=set(); mst=[]; h=[(0,None,s)]
    while h:
        w,u,v=heapq.heappop(h)
        if v in vis: continue
        vis.add(v)
        if u is not None: mst.append((u,v,w))
        for nb,wt in adj[v]:
            if nb not in vis: heapq.heappush(h,(wt,v,nb))
    return mst, sum(c for _,_,c in mst)

# ── Recursion helpers ─────────────────────────────────────────────────
class Profiler:
    def __init__(self,fn): self.fn=fn; self.calls=0; self._d=0; self.md=0
    def __call__(self,*a,**kw):
        self.calls+=1; self._d+=1; self.md=max(self.md,self._d)
        r=self.fn(*a,**kw); self._d-=1; return r
    def reset(self): self.calls=self._d=self.md=0

print("✅  Part 4 setup complete — all algorithms ready.")
print(f"    {len(ALGORITHMS)} sorting algorithms | Kruskal + Prim | Recursion profiler")


---
## Chart 1 — Sorting Benchmark Bar Charts

In [ ]:
# =========================================================
#  CELL 2 - CHART 1: SORTING BENCHMARK BARS
# =========================================================
import warnings
warnings.filterwarnings("ignore")

N = 1000
random.seed(0)
data = [random.randint(0, 9999) for _ in range(N)]

names_list = []; times_list = []; comps_list = []; swaps_list = []
for name, fn in ALGORITHMS:
    t0 = time.perf_counter()
    _, c, s = fn(data[:])
    names_list.append(name)
    times_list.append((time.perf_counter()-t0)*1000)
    comps_list.append(c)
    swaps_list.append(s)

short = [n.replace(" Sort","").replace("-"," ") for n in names_list]

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle(f"Chart 1 - Sorting Benchmark  (n = {N})",
             fontsize=13, fontweight="bold", y=1.01)

for ax, vals, fmt, title in zip(
        axes,
        [times_list, comps_list, swaps_list],
        [".3f", ",d", ",d"],
        ["Execution Time (ms)", "Comparisons", "Swaps"]):
    pairs = sorted(zip(vals, short, COLORS), reverse=True)
    v, l, c = zip(*pairs)
    bars = ax.barh(l, v, color=c, edgecolor="white", lw=1.2, height=0.6)
    ax.bar_label(bars,
                 labels=[format(x, fmt) for x in v],
                 padding=4, fontsize=8)
    ax.set_title(title, pad=8)
    ax.spines[["top","right"]].set_visible(False)
    ax.tick_params(axis="y", labelsize=9)

plt.tight_layout()
plt.savefig("/tmp/p4_chart1_bars.png", bbox_inches="tight", dpi=110)
plt.show()
print("Chart 1: benchmark bars - done")


---
## Chart 2 — Sorting Scalability

In [ ]:
# ═══════════════════════════════════════════════════════════════════════
#  CELL 3 — CHART 2: SCALABILITY CURVE
#  Time vs dataset size for all 8 algorithms
# ═══════════════════════════════════════════════════════════════════════
SIZES = [50, 100, 250, 500, 1000, 2000]
scale = {name: [] for name, _ in ALGORITHMS}

print("Running scalability sweep...")
for sz in SIZES:
    d = [random.randint(0, 9999) for _ in range(sz)]
    for name, fn in ALGORITHMS:
        t0 = time.perf_counter()
        fn(d[:])
        scale[name].append((time.perf_counter()-t0)*1000)
    print(f"  n={sz:>5} ✓")

fig, ax = plt.subplots(figsize=(12, 5))
for i, (name, _) in enumerate(ALGORITHMS):
    ax.plot(SIZES, scale[name], marker="o", label=name,
            color=COLORS[i], lw=2, ms=6, alpha=0.9)
ax.set_xlabel("Dataset Size (n)")
ax.set_ylabel("Time (ms)")
ax.set_title("Chart 2 — Scalability: Time vs Dataset Size", pad=10)
ax.legend(loc="upper left", fontsize=9, framealpha=0.9, ncol=2, columnspacing=0.8)
ax.spines[["top","right"]].set_visible(False)
plt.tight_layout()
plt.savefig("/tmp/p4_chart2_scalability.png", bbox_inches="tight", dpi=110)
plt.show()
print("Chart 2: scalability — done")


---
## Chart 3 — Best / Average / Worst Case Heatmap

In [ ]:
# ═══════════════════════════════════════════════════════════════════════
#  CELL 4 — CHART 3: BEST / AVERAGE / WORST CASE HEATMAP
# ═══════════════════════════════════════════════════════════════════════
n_h = 500
cases = {
    "Best\n(sorted)":    list(range(n_h)),
    "Average\n(random)": [random.randint(0,9999) for _ in range(n_h)],
    "Worst\n(reversed)": list(range(n_h, 0, -1)),
}
M = np.array([
    [(lambda t0, fn=fn, d=d: (time.perf_counter()-t0)*1000)(time.perf_counter(), fn(d[:]))
     for d in cases.values()]
    for name, fn in ALGORITHMS
])

fig, ax = plt.subplots(figsize=(9, 6))
im = ax.imshow(M, cmap="YlOrRd", aspect="auto")
ax.set_xticks(range(len(cases))); ax.set_xticklabels(list(cases.keys()), fontsize=10)
ax.set_yticks(range(len(ALGORITHMS))); ax.set_yticklabels([n for n,_ in ALGORITHMS], fontsize=9)
for i in range(len(ALGORITHMS)):
    for j in range(len(cases)):
        v = M[i,j]
        col = "white" if v > M.max()*0.55 else "black"
        ax.text(j, i, f"{v:.2f}", ha="center", va="center",
                color=col, fontsize=9, fontweight="bold")
plt.colorbar(im, ax=ax, label="Time (ms)", shrink=0.8)
ax.set_title(f"Chart 3 — Best / Average / Worst Case Heatmap (n={n_h})", pad=10)
plt.tight_layout()
plt.savefig("/tmp/p4_chart3_heatmap.png", bbox_inches="tight", dpi=110)
plt.show()
print("Chart 3: heatmap — done")


---
## Chart 4 — MST Graph Visualization

In [ ]:
# =========================================================
#  CELL 5 - CHART 4: MST GRAPH VISUALIZATION
# =========================================================

VERTS4 = [1,2,3,4,5,6]
EDGES4 = [(1,2,4),(1,3,3),(2,3,5),(2,4,6),(3,4,7),
          (3,5,8),(4,5,9),(4,6,5),(5,6,6)]

k_mst4, k_cost4 = kruskal(VERTS4, EDGES4)
p_mst4, p_cost4 = prim(VERTS4, EDGES4, start=1)

G4 = nx.Graph(); G4.add_nodes_from(VERTS4)
for u,v,w in EDGES4: G4.add_edge(u,v,weight=w)
POS4 = nx.spring_layout(G4, seed=42)

def draw_p4(ax, title, mst_edges=None):
    mst_set = set()
    if mst_edges:
        for u,v,_ in mst_edges:
            mst_set.add((u,v)); mst_set.add((v,u))
    ecols = ["#2A9D8F" if (u,v) in mst_set else "#CBD5E1"
             for u,v in G4.edges()]
    ewids = [3.5 if (u,v) in mst_set else 1.0
             for u,v in G4.edges()]
    nx.draw_networkx_nodes(G4, POS4, ax=ax,
                           node_color="#264653", node_size=700)
    nx.draw_networkx_labels(G4, POS4, ax=ax,
                            font_color="white", font_size=12)
    nx.draw_networkx_edges(G4, POS4, ax=ax,
                           edge_color=ecols, width=ewids, alpha=0.9)
    nx.draw_networkx_edge_labels(
        G4, POS4,
        edge_labels=nx.get_edge_attributes(G4,"weight"),
        ax=ax, font_size=8)
    ax.set_title(title, pad=10); ax.axis("off")

fig, axes = plt.subplots(1, 3, figsize=(17, 5))
fig.suptitle("Chart 4 - MST Visualization: Kruskal vs Prim",
             fontsize=13, fontweight="bold", y=1.02)
draw_p4(axes[0], "Full Graph (all edges)")
draw_p4(axes[1], f"Kruskal MST  (cost = {k_cost4})", k_mst4)
draw_p4(axes[2], f"Prim MST     (cost = {p_cost4})", p_mst4)
leg = [mpatches.Patch(color="#2A9D8F",label="MST edge"),
       mpatches.Patch(color="#CBD5E1",label="Non-MST edge")]
fig.legend(handles=leg, loc="lower center", ncol=2, frameon=False,
           fontsize=10, bbox_to_anchor=(0.5,-0.04))
plt.tight_layout()
plt.savefig("/tmp/p4_chart4_mst.png", bbox_inches="tight", dpi=110)
plt.show()
print(f"Chart 4: MST graph - Kruskal={k_cost4}, Prim={p_cost4}")


---
## Chart 5 — Recursive Call Count Growth

In [ ]:
# =========================================================
#  CELL 6 - CHART 5: RECURSIVE CALL COUNT GROWTH
# =========================================================

def _ff(n): return 1 if n<=1 else n*_ff(n-1)
def _bb(n): return n if n<=1 else _bb(n-1)+_bb(n-2)
def _mm(n, c=None):
    if c is None: c={}
    if n in c: return c[n]
    if n<=1: return n
    c[n]=_mm(n-1,c)+_mm(n-2,c); return c[n]

pFF=Profiler(_ff); _ff=pFF
pBB=Profiler(_bb); _bb=pBB
pMM=Profiler(_mm); _mm=pMM

ns4=list(range(1,17)); fc4=[]; bc4=[]; mc4=[]
for n in ns4:
    pFF.reset(); _ff(n); fc4.append(pFF.calls)
    pBB.reset(); _bb(n); bc4.append(pBB.calls)
    pMM.reset(); _mm(n); mc4.append(pMM.calls)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("Chart 5 - Recursive Call Count Growth",
             fontsize=13, fontweight="bold", y=1.01)
for ax, logy in zip(axes, [False, True]):
    ax.plot(ns4, fc4, "o-", color="#457B9D", lw=2.5, ms=7,
            label="Factorial  O(n)")
    ax.plot(ns4, mc4, "s--", color="#2A9D8F", lw=2.5, ms=7,
            label="Fib Memo  O(n)")
    ax.plot(ns4, bc4, "^-", color="#E63946", lw=2.5, ms=7,
            label="Fib Naive  O(2^n)")
    if logy:
        ax.set_yscale("log"); ax.set_title("Log Scale", pad=8)
    else:
        ax.set_title("Linear Scale", pad=8)
    ax.set_xlabel("n")
    ax.set_ylabel("Total Calls" + (" (log)" if logy else ""))
    ax.legend(fontsize=10)
    ax.spines[["top","right"]].set_visible(False)
plt.tight_layout()
plt.savefig("/tmp/p4_chart5_recursion.png", bbox_inches="tight", dpi=110)
plt.show()
print("Chart 5: recursion growth - done")


---
## Chart 6 — Tower of Hanoi Exponential Growth

In [ ]:
# ═══════════════════════════════════════════════════════════════════════
#  CELL 7 — CHART 6: TOWER OF HANOI EXPONENTIAL GROWTH
# ═══════════════════════════════════════════════════════════════════════
def hanoi_moves(n):
    m=[0]
    def h(n,s,a,d):
        if n==1: m[0]+=1; return
        h(n-1,s,d,a); m[0]+=1; h(n-1,a,s,d)
    h(n,"A","B","C"); return m[0]

disks   = list(range(1, 17))
actual  = [hanoi_moves(d) for d in disks]
theory  = [2**d - 1 for d in disks]

fig, ax = plt.subplots(figsize=(11, 5))
ax.bar(disks, actual, color="#9B5DE5", alpha=0.85, label="Actual moves  (counted)")
ax.plot(disks, theory, "r--o", lw=1.8, ms=5, label="2ⁿ − 1  (formula)")
ax.set_xlabel("Number of Disks")
ax.set_ylabel("Total Moves Required")
ax.set_title("Chart 6 — Tower of Hanoi: Move Count = 2ⁿ − 1", pad=10)
ax.legend(fontsize=10)
ax.spines[["top","right"]].set_visible(False)
plt.tight_layout()
plt.savefig("/tmp/p4_chart6_hanoi.png", bbox_inches="tight", dpi=110)
plt.show()
print(f"Chart 6: Hanoi growth — formula verified: {actual == theory}")
print()
print("All 6 charts complete.")


---
## Chart 7 — Recursion Tree Diagrams

In [ ]:
# =========================================================
#  CELL 8 - CHART 7: RECURSION TREE DIAGRAMS
#  Left  : Factorial call chain  (linear, O(n) depth)
#  Right : Fibonacci binary tree (branching, O(2^n) calls)
#  Bonus : "Recursion tree visualization" per spec
# =========================================================

fig, axes = plt.subplots(1, 2, figsize=(16, 7))
fig.suptitle("Chart 7 - Recursion Tree Visualization",
             fontsize=13, fontweight="bold", y=1.01)

# ── Left: Factorial call chain ────────────────────────────────────────
ax = axes[0]
ax.axis("off")
ax.set_title("Factorial(5) - Linear Call Chain  O(n) depth", pad=10)

N_FACT = 5
CALL_COLORS = ["#E63946","#457B9D","#2A9D8F","#E9C46A","#F4A261","#264653"]

for i, val in enumerate(range(N_FACT, 0, -1)):
    y   = (N_FACT - i) * 1.2 + 0.5
    col = CALL_COLORS[i % len(CALL_COLORS)]
    rect = plt.Rectangle((0.5, y - 0.4), 9.0, 0.8,
                          facecolor=col + "33", edgecolor=col, linewidth=2,
                          transform=ax.transData)
    ax.add_patch(rect)
    if val == 1:
        label = f"factorial(1)  ->  base case: return 1"
    else:
        label = f"factorial({val})  ->  {val} x factorial({val-1})"
    ax.text(5.0, y, label, ha="center", va="center",
            fontsize=10, fontfamily="monospace", fontweight="bold")
    # Arrow between boxes
    if i < N_FACT - 1:
        ax.annotate("",
                    xy=(5.0, y - 0.4),
                    xytext=(5.0, y - 0.4 - 0.4),
                    arrowprops=dict(arrowstyle="->", color="#555", lw=1.5))

# Result box
result_val = 1
for x in range(1, N_FACT + 1): result_val *= x
ax.text(5.0, 0.0, f"Result = {result_val}", ha="center", va="center",
        fontsize=12, fontfamily="monospace", fontweight="bold",
        bbox=dict(boxstyle="round,pad=0.5", facecolor="#4ADE80",
                  edgecolor="#16A34A", lw=2.5))
ax.set_xlim(0, 10)
ax.set_ylim(-0.8, (N_FACT + 1) * 1.2 + 0.5)


# ── Right: Fibonacci binary tree ─────────────────────────────────────
ax2 = axes[1]
ax2.axis("off")
N_FIB = 5
ax2.set_title(f"Fibonacci({N_FIB}) - Binary Tree  O(2^n) calls", pad=10)

NODE_COL  = ["#E63946","#457B9D","#2A9D8F","#E9C46A","#F4A261"]
BASE_COL  = "#9B5DE5"
node_list = []
edge_list = []

def collect_nodes(val, depth, x, spread):
    if depth > 4:
        return
    node_list.append((x, -depth * 1.4, val, depth))
    if val > 1:
        lx = x - spread
        rx = x + spread
        edge_list.append(((x, -depth * 1.4), (lx, -(depth + 1) * 1.4)))
        edge_list.append(((x, -depth * 1.4), (rx, -(depth + 1) * 1.4)))
        collect_nodes(val - 1, depth + 1, lx, spread * 0.5)
        collect_nodes(val - 2, depth + 1, rx, spread * 0.5)

collect_nodes(N_FIB, 0, 0, 4.0)

for (x1, y1), (x2, y2) in edge_list:
    ax2.plot([x1, x2], [y1, y2], color="#CBD5E1", lw=1.5, zorder=1)

for x, y, val, depth in node_list:
    is_base = val <= 1
    col = BASE_COL if is_base else NODE_COL[depth % len(NODE_COL)]
    circle = plt.Circle((x, y), 0.45,
                         facecolor=col + "55", edgecolor=col, lw=2.0, zorder=2)
    ax2.add_patch(circle)
    ax2.text(x, y, f"f({val})", ha="center", va="center",
             fontsize=8, fontfamily="monospace", fontweight="bold", zorder=3)

import matplotlib.patches as _mp
legend_handles = [
    _mp.Patch(facecolor=BASE_COL + "55", edgecolor=BASE_COL,
              label="Base case  f(0) or f(1)"),
    _mp.Patch(facecolor="#E63946" + "55", edgecolor="#E63946",
              label="Recursive call"),
]
ax2.legend(handles=legend_handles, loc="lower center",
           fontsize=9, framealpha=0.9, bbox_to_anchor=(0.5, -0.05))
ax2.set_xlim(-9, 9)
ax2.set_ylim(-N_FIB * 1.4 - 1.0, 1.2)

plt.tight_layout()
plt.savefig("/tmp/p4_chart7_recursion_tree.png", bbox_inches="tight", dpi=110)
plt.show()
print(f"Chart 7: recursion tree diagram - done")
print(f"  Factorial({N_FACT}): linear chain, {N_FACT} frames, depth = {N_FACT}")
print(f"  Fibonacci({N_FIB}): binary tree, calls shown up to depth 4")


---
## Chart 8 — Sorting Metrics Breakdown

In [ ]:
# =========================================================
#  CELL 9 - CHART 8: SORTING METRICS BREAKDOWN
#  Separate charts for Comparisons and Swaps (not just time)
#  Helps explain WHY some O(n log n) algorithms differ in practice
# =========================================================

random.seed(0)
_d2 = [random.randint(0, 9999) for _ in range(1000)]

names8  = []
times8  = []
comps8  = []
swaps8  = []
for name, fn in ALGORITHMS:
    t0 = time.perf_counter()
    _, c, s = fn(_d2[:])
    names8.append(name.replace(" Sort", "").replace("-", " "))
    times8.append((time.perf_counter() - t0) * 1000)
    comps8.append(c)
    swaps8.append(s)

x8 = range(len(names8))

fig, axes = plt.subplots(1, 3, figsize=(17, 5))
fig.suptitle("Chart 8 - Sorting Metrics Breakdown at n=1000",
             fontsize=13, fontweight="bold", y=1.01)

# Times
bars0 = axes[0].bar(x8, times8, color=COLORS, edgecolor="white", lw=1, alpha=0.9)
axes[0].set_title("Execution Time (ms)", pad=8)
axes[0].set_ylabel("ms")
axes[0].set_xticks(x8); axes[0].set_xticklabels(names8, rotation=30, ha="right", fontsize=8)
axes[0].bar_label(bars0, labels=[f"{v:.2f}" for v in times8], padding=3, fontsize=7)
axes[0].spines[["top", "right"]].set_visible(False)

# Comparisons
bars1 = axes[1].bar(x8, comps8, color=COLORS, edgecolor="white", lw=1, alpha=0.9)
axes[1].set_title("Number of Comparisons", pad=8)
axes[1].set_ylabel("count")
axes[1].set_xticks(x8); axes[1].set_xticklabels(names8, rotation=30, ha="right", fontsize=8)
axes[1].bar_label(bars1,
                  labels=[f"{int(v/1000)}k" if v >= 1000 else str(int(v)) for v in comps8],
                  padding=3, fontsize=7)
axes[1].spines[["top", "right"]].set_visible(False)

# Swaps
bars2 = axes[2].bar(x8, swaps8, color=COLORS, edgecolor="white", lw=1, alpha=0.9)
axes[2].set_title("Number of Swaps", pad=8)
axes[2].set_ylabel("count")
axes[2].set_xticks(x8); axes[2].set_xticklabels(names8, rotation=30, ha="right", fontsize=8)
axes[2].bar_label(bars2,
                  labels=[f"{int(v/1000)}k" if v >= 1000 else str(int(v)) for v in swaps8],
                  padding=3, fontsize=7)
axes[2].spines[["top", "right"]].set_visible(False)

plt.tight_layout()
plt.savefig("/tmp/p4_chart8_metrics.png", bbox_inches="tight", dpi=110)
plt.show()
print("Chart 8: sorting metrics breakdown - done")

# Print key observations
print()
print("  Key observations:")
max_c_idx = comps8.index(max(comps8))
min_c_idx = comps8.index(min(c for c in comps8 if c > 0))
max_s_idx = swaps8.index(max(swaps8))
print(f"  Most comparisons  : {names8[max_c_idx]}  ({comps8[max_c_idx]:,})")
print(f"  Least comparisons : {names8[min_c_idx]}  ({comps8[min_c_idx]:,})")
print(f"  Most swaps        : {names8[max_s_idx]}  ({swaps8[max_s_idx]:,})")
no_swap = [names8[i] for i,s in enumerate(swaps8) if s == 0]
print(f"  Zero swaps        : {', '.join(no_swap)}")


---
## Chart 9 — MST Cost Comparison

In [ ]:
# =========================================================
#  CELL 10 - CHART 9: MST COST COMPARISON
#  Runs Kruskal and Prim on 10 random graphs of increasing size.
#  Confirms both algorithms always find the SAME minimum cost.
# =========================================================

import numpy as _np9
_np9.random.seed(7)

def random_graph(n_verts, n_extra=5):
    """Generate a connected random weighted graph."""
    verts = list(range(1, n_verts + 1))
    edges_dict = {}
    # Spanning chain ensures connectivity
    for i in range(1, n_verts):
        w = int(_np9.random.randint(1, 21))
        edges_dict[(i, i + 1)] = w
    # Extra random edges
    attempts = 0
    while len(edges_dict) < n_verts - 1 + n_extra and attempts < 500:
        attempts += 1
        u = int(_np9.random.randint(1, n_verts + 1))
        v = int(_np9.random.randint(1, n_verts + 1))
        if u != v:
            k = (min(u, v), max(u, v))
            if k not in edges_dict:
                edges_dict[k] = int(_np9.random.randint(1, 21))
    return verts, [(u, v, w) for (u, v), w in edges_dict.items()]

GRAPH_SIZES   = [4, 5, 6, 7, 8, 9, 10, 12, 14, 16]
kruskal_costs = []
prim_costs    = []
agree_all     = True

print("  MST Cost Comparison — 10 random graphs:")
print(f"  {'Vertices':>10}  {'Edges':>7}  {'Kruskal':>10}  {'Prim':>8}  {'Agree?'}")
print("  " + "-" * 52)

for n_v in GRAPH_SIZES:
    verts, edges = random_graph(n_v, n_extra=n_v)
    _, kc = kruskal(verts, edges)
    _, pc = prim(verts, edges)
    agree = kc == pc
    if not agree: agree_all = False
    kruskal_costs.append(kc)
    prim_costs.append(pc)
    print(f"  {n_v:>10}  {len(edges):>7}  {kc:>10}  {pc:>8}  {'YES' if agree else 'NO'}")

print()
print(f"  All costs agree: {'YES - both algorithms always optimal!' if agree_all else 'NO - check implementation'}")

fig2, (ax_a, ax_b) = plt.subplots(1, 2, figsize=(14, 5))
fig2.suptitle("Chart 9 - MST Cost: Kruskal vs Prim (10 random graphs)",
              fontsize=13, fontweight="bold", y=1.01)

# Line plot: costs
ax_a.plot(GRAPH_SIZES, kruskal_costs, "o-", color="#457B9D",
          lw=2.5, ms=8, label="Kruskal's")
ax_a.plot(GRAPH_SIZES, prim_costs,    "s--", color="#2A9D8F",
          lw=2.5, ms=8, label="Prim's")
ax_a.set_xlabel("Number of Vertices")
ax_a.set_ylabel("MST Total Cost")
ax_a.set_title("MST Cost vs Graph Size", pad=8)
ax_a.legend(fontsize=10)
ax_a.spines[["top", "right"]].set_visible(False)

# Bar chart: side by side
x9  = range(len(GRAPH_SIZES))
ax_b.bar([xx - 0.2 for xx in x9], kruskal_costs,
         width=0.35, color="#457B9D", label="Kruskal's", alpha=0.9)
ax_b.bar([xx + 0.2 for xx in x9], prim_costs,
         width=0.35, color="#2A9D8F", label="Prim's", alpha=0.9)
ax_b.set_xticks(list(x9))
ax_b.set_xticklabels([str(s) for s in GRAPH_SIZES], fontsize=9)
ax_b.set_xlabel("Vertices")
ax_b.set_ylabel("MST Total Cost")
ax_b.set_title("Side-by-Side Comparison", pad=8)
ax_b.legend(fontsize=10)
ax_b.spines[["top", "right"]].set_visible(False)

plt.tight_layout()
plt.savefig("/tmp/p4_chart9_mst_costs.png", bbox_inches="tight", dpi=110)
plt.show()
print("Chart 9: MST cost comparison - done")
